# Part 9 — Client Deployment & Consulting (FDE module)

*CinemaStream — The Forward Deployed Engineer's Handbook*

---

In [ ]:
# ── CinemaStream: one-time setup ──────────────────────────────────────────────
# Run this cell FIRST if you are on Google Colab or a fresh local environment.
# Skip it if you have already cloned the repo and installed requirements.
#
# !pip install -r requirements.txt
# !git clone https://github.com/YOUR_ORG/cinemastream.git
# import os; os.chdir("cinemastream")
# ─────────────────────────────────────────────────────────────────────────────
# Ensure the canonical dataset exists (deterministic; safe to re-run).
try:
    from cinemastream.scripts.generate_data import generate
    generate()
except ModuleNotFoundError:
    print("Run the clone/cd lines above first (Colab), then re-run this cell.")

## Chapters in this notebook

- [Chapter 86: Discovery Interviews — Finding the Real Requirement](#chapter_86_discovery_interviews_finding_the_real_requirement)
- [Chapter 87: User Stories and Acceptance Criteria — Defining "Done" Before You Build](#chapter_87_user_stories_and_acceptance_criteria_defining_done_before_you_build)
- [Chapter 88: Prioritization — RICE, MoSCoW, and the Power of "Won't"](#chapter_88_prioritization_rice_moscow_and_the_power_of_won_t)
- [Chapter 89: Scope Management — Change Requests, Saying No, and Protecting the MVP](#chapter_89_scope_management_change_requests_saying_no_and_protecting_the_mvp)
- [Chapter 90: The PoV-PoC-MVP Lifecycle — Proving Value Before Proving Feasibility Before Earning Adoption](#chapter_90_the_pov_poc_mvp_lifecycle_proving_value_before_proving_feasibility_before_earning_adoption)
- [Chapter 91: Demo Storytelling — Making Someone Who Wasn't in the Room Feel It](#chapter_91_demo_storytelling_making_someone_who_wasn_t_in_the_room_feel_it)
- [Chapter 92: Adoption Strategy — Making What You Built Actually Get Used](#chapter_92_adoption_strategy_making_what_you_built_actually_get_used)

---

# Chapter 86: Discovery Interviews — Finding the Real Requirement

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 The five questions you ask of every data source

In [ ]:
DISCOVERY_QUESTIONS = [
    "What kind is it?    structured (rows/API) or unstructured (text/PDF)?",
    "Who OWNS it?        the accountable team (not a person -- people leave)",
    "Who may SEE it?     the access-control boundary",
    "How often CHANGES?  drives the freshness/refresh strategy",
    "What's the REAL need? the job behind the stated ask",
]
for i, q in enumerate(DISCOVERY_QUESTIONS, 1):
    print(f"  Q{i}. {q}")

```
  Q1. What kind is it?    structured (rows/API) or unstructured (text/PDF)?
  Q2. Who OWNS it?        the accountable team (not a person -- people leave)
  Q3. Who may SEE it?     the access-control boundary
  Q4. How often CHANGES?  drives the freshness/refresh strategy
  Q5. What's the REAL need? the job behind the stated ask
```

### 2.2 Categorize: structured → MCP, unstructured → RAG

In [ ]:
def build_path(kind):
    return "MCP tool (Claude calls it live)" if kind == "structured" \
        else "RAG pipeline (embed + retrieve)"

abstract_sources = [
    ("Salesforce CRM (API)", "structured"),
    ("Folder of contract PDFs", "unstructured"),
    ("Postgres orders table", "structured"),
    ("Slack #support history", "unstructured"),
]
for name, kind in abstract_sources:
    print(f"  {name:28s} [{kind:12s}] -> {build_path(kind)}")

```
  Salesforce CRM (API)         [structured  ] -> MCP tool (Claude calls it live)
  Folder of contract PDFs      [unstructured] -> RAG pipeline (embed + retrieve)
  Postgres orders table        [structured  ] -> MCP tool (Claude calls it live)
  Slack #support history       [unstructured] -> RAG pipeline (embed + retrieve)
```

### 2.3 Laddering: the stated ask is never the real need

In [ ]:
ladder = [
    ("STATED ASK", "I want a dashboard of everything."),
    ("why?", "So I can stop asking the data team for numbers."),
    ("why?", "Because they take two days and I decide in two hours."),
    ("REAL NEED", "Self-serve answers to 3 recurring questions, fast."),
]
for label, text in ladder:
    print(f"  {label:11s}: {text}")

```
  STATED ASK : I want a dashboard of everything.
  why?       : So I can stop asking the data team for numbers.
  why?       : Because they take two days and I decide in two hours.
  REAL NEED  : Self-serve answers to 3 recurring questions, fast.
```

## 3. CinemaStream in Practice — On Loan at FilmiBox

In [ ]:
from dataclasses import dataclass
from enum import Enum

class Kind(Enum):
    STRUCTURED = "structured"      # rows/API -> MCP tool
    UNSTRUCTURED = "unstructured"  # text/PDF -> RAG pipeline

@dataclass
class DataSource:
    name: str
    kind: Kind
    owner: str            # accountable team
    access: str           # who may see it
    update_freq: str      # freshness strategy
    real_need: str        # the job, not the stated ask

    def build_path(self):
        return "MCP tool" if self.kind is Kind.STRUCTURED else "RAG pipeline"

In [ ]:
# FILMIBOX_SOURCES + summarize() are the deliverable, built from the interviews
from filmibox.discovery.data_sources import summarize, FILMIBOX_SOURCES
summarize(FILMIBOX_SOURCES)

```
FilmiBox discovery map: 5 data sources

source                                   build path   access
--------------------------------------------------------------------------------------------
Subscriber database (Postgres)           MCP tool     PII -- restricted to support + eng
Content catalog (titles API)             MCP tool     internal (all staff)
Support tickets (Zendesk export)         RAG pipeline PII -- support + ops only
Company wiki + runbooks (Hindi/English)  RAG pipeline internal (all staff)
Revenue reports (PDF/spreadsheet)        RAG pipeline CONFIDENTIAL -- founders only

-> 2 structured sources become MCP tools, 3 unstructured become RAG pipelines.

ACCESS-CONTROL FLAG: 1 source(s) are founder-confidential and must NOT enter the all-staff assistant:
   - Revenue reports (PDF/spreadsheet) (owner: Anjali (CFO duties))
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 87: User Stories and Acceptance Criteria — Defining "Done" Before You Build

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 The user story template

In [ ]:
def story(role, want, so_that):
    return f"As a {role}, I want to {want}, so that {so_that}."

print(story("warehouse manager", "see today's low-stock items",
            "I can reorder before we run out"))

```
As a warehouse manager, I want to see today's low-stock items, so that I can reorder before we run out.
```

### 2.2 INVEST: the quality bar

In [ ]:
INVEST = [
    ("Independent", "can be built without waiting on another story"),
    ("Negotiable", "a conversation, not a rigid spec"),
    ("Valuable", "delivers value to a user (the 'so that')"),
    ("Estimable", "small enough that you can estimate the effort"),
    ("Small", "ONE capability -- fits in a few days, not a quarter"),
    ("Testable", "has acceptance criteria -- you can prove it's done"),
]
for word, meaning in INVEST:
    print(f"  {word[0]} - {word:12s} {meaning}")

```
  I - Independent  can be built without waiting on another story
  N - Negotiable   a conversation, not a rigid spec
  V - Valuable     delivers value to a user (the 'so that')
  E - Estimable    small enough that you can estimate the effort
  S - Small        ONE capability -- fits in a few days, not a quarter
  T - Testable     has acceptance criteria -- you can prove it's done
```

### 2.3 Given-When-Then: the definition of done

In [ ]:
def gwt(given, when, then):
    return f"GIVEN {given}\n  WHEN {when}\n  THEN {then}"

print(gwt("a manager is logged in",
          "they ask for today's low-stock items",
          "the system lists every SKU below its reorder threshold"))

```
GIVEN a manager is logged in
  WHEN they ask for today's low-stock items
  THEN the system lists every SKU below its reorder threshold
```

### 2.4 A good story versus a wish

In [ ]:
good = story("support agent", "look up a subscriber's recent charges",
             "I can answer billing questions without escalating")
print(f"GOOD: {good}")
print("BAD:  As a founder, I want full visibility into everything, so that ... "
      "(no benefit, huge scope, no acceptance criteria)")

```
GOOD: As a support agent, I want to look up a subscriber's recent charges, so that I can answer billing questions without escalating.
BAD:  As a founder, I want full visibility into everything, so that ... (no benefit, huge scope, no acceptance criteria)
```

## 3. CinemaStream in Practice — On Loan at FilmiBox

In [ ]:
from dataclasses import dataclass, field

@dataclass
class AcceptanceCriterion:
    given: str; when: str; then: str          # Given-When-Then = definition of done

@dataclass
class UserStory:
    role: str; want: str; so_that: str; access: str
    criteria: list = field(default_factory=list)
    def story(self):
        return f"As a {self.role}, I want to {self.want}, so that {self.so_that}."
    def invest(self):
        return {"Valuable": bool(self.so_that),
                "Estimable": len(self.want) < 120,
                "Small": " and " not in self.want.lower(),   # one capability
                "Testable": len(self.criteria) >= 1}

In [ ]:
# STORIES + main() are the deliverable, built from the five discovery needs
from filmibox.discovery.user_stories import STORIES, main
main()

```
FilmiBox backlog -- user stories vs the INVEST bar

Story 1 [support + eng]  -- INVEST: OK
  As a support agent, I want to look up a specific subscriber's recent charges by email, so that I can answer 'why was I charged twice' without escalating to engineering.
  - GIVEN a subscriber email that exists
      WHEN the agent asks for that user's charges in the last 60 days
      THEN the assistant returns the dated charge list from the live database

Story 2 [support + ops]  -- INVEST: OK
  As a support lead, I want to see this week's top recurring complaint themes, so that I can flag an emerging bug before it reaches more users.
  - GIVEN the last 7 days of support tickets are indexed
      WHEN the lead asks for the top complaint themes
      THEN the assistant returns 3-5 themes ranked by ticket count

Story 3 [all staff]  -- INVEST: OK
  As a content manager, I want to ask how many titles we hold in a given genre, so that I can avoid licensing duplicates.
  - GIVEN the content catalog is reachable
      WHEN the manager asks 'how many Tamil thrillers from the last 2 years?'
      THEN the assistant returns the count and titles from the live catalog

Story 4 [all staff]  -- INVEST: OK
  As a new hire, I want to ask how to perform a common procedure like issuing a refund, so that I don't interrupt senior staff for things the wiki already answers.
  - GIVEN the bilingual (Hindi/English) wiki is indexed
      WHEN the new hire asks 'how do I issue a refund?' in either language
      THEN the assistant answers from the wiki and cites the source page

Story 5 [founders]  -- INVEST: NEEDS WORK (1/4)
  As a founder, I want to get full visibility into everything happening across the company and all our metrics and all the data in one place, so that .
  FAILS: Valuable, Small, Testable
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 88: Prioritization — RICE, MoSCoW, and the Power of "Won't"

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 RICE: one score for value-per-effort

In [ ]:
def rice(reach, impact, confidence, effort):
    return (reach * impact * confidence) / effort   # (R x I x C) / E

# impact: 3=massive 2=high 1=medium 0.5=low ; confidence: 1.0/0.8/0.5
features = [
    ("Quick CSV export", 200, 0.5, 1.0, 0.5),
    ("AI recommender",   500, 3,   0.5, 8.0),
    ("Bulk edit tool",   120, 1,   0.8, 1.0),
]
print(f"{'feature':18s} {'R':>4} {'I':>4} {'C':>4} {'E':>4} {'RICE':>7}")
for name, R, I, C, E in features:
    print(f"{name:18s} {R:>4} {I:>4} {C:>4} {E:>4} {rice(R,I,C,E):>7.1f}")

```
feature               R    I    C    E    RICE
Quick CSV export    200  0.5  1.0  0.5   200.0
AI recommender      500    3  0.5  8.0    93.8
Bulk edit tool      120    1  0.8  1.0    96.0
```

### 2.2 The Effort divisor is the point

In [ ]:
print(f"  AI recommender, effort 8.0 wks -> RICE {rice(500,3,0.5,8.0):.1f}")
print(f"  same feature, if effort were 4.0 wks -> RICE {rice(500,3,0.5,4.0):.1f}")

```
  AI recommender, effort 8.0 wks -> RICE 93.8
  same feature, if effort were 4.0 wks -> RICE 187.5
```

### 2.3 MoSCoW: turn the ranking into a committed scope

In [ ]:
MOSCOW = {
    "Must":   "the release fails without it -- non-negotiable",
    "Should": "important, painful to omit, but the release survives",
    "Could":  "nice-to-have, first to cut when time runs short",
    "Won't":  "explicitly OUT of this phase (not 'never' -- 'not now')",
}
for bucket, meaning in MOSCOW.items():
    print(f"  {bucket:7s} {meaning}")

```
  Must    the release fails without it -- non-negotiable
  Should  important, painful to omit, but the release survives
  Could   nice-to-have, first to cut when time runs short
  Won't   explicitly OUT of this phase (not 'never' -- 'not now')
```

## 3. CinemaStream in Practice — On Loan at FilmiBox

In [ ]:
from dataclasses import dataclass

@dataclass
class Story:
    name: str; reach: int; impact: float; confidence: float; effort: float; moscow: str
    def rice(self):
        return (self.reach * self.impact * self.confidence) / self.effort

BACKLOG = [
    Story("1. Support billing lookup",   reach=40, impact=2, confidence=0.9, effort=1.0, moscow="Must"),
    Story("2. Support complaint themes", reach=20, impact=2, confidence=0.7, effort=1.5, moscow="Should"),
    Story("3. Content catalog count",    reach=10, impact=1, confidence=0.9, effort=0.5, moscow="Could"),
    Story("4. New-hire refund wiki",     reach=60, impact=1, confidence=0.8, effort=1.5, moscow="Should"),
    Story("5. Founder churn-adj MRR",    reach=2,  impact=3, confidence=0.8, effort=2.0, moscow="Won't (this phase)"),
]

In [ ]:
ranked = sorted(BACKLOG, key=lambda s: s.rice(), reverse=True)
for s in ranked:
    print(f"{s.name:30s} RICE {s.rice():>6.1f}  {s.moscow}")

```
1. Support billing lookup      RICE   72.0  Must
4. New-hire refund wiki        RICE   32.0  Should
2. Support complaint themes    RICE   18.7  Should
3. Content catalog count       RICE   18.0  Could
5. Founder churn-adj MRR       RICE    2.4  Won't (this phase)
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 89: Scope Management — Change Requests, Saying No, and Protecting the MVP

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 The three-bucket change request response

In [ ]:
CHANGE_BUCKETS = {
    "YES_TRADE":    "In scope or genuine must-have: accept, but remove something of equal cost",
    "NO_PHASE":     "Valuable but not this phase: decline with a named future home",
    "DEFER_ASSESS": "Unclear value or effort: defer until scoped; don't absorb unknowns",
}

def classify_change(description, is_in_scope_extension, effort_weeks, phase_capacity_remaining):
    """
    Returns a bucket + one-sentence rationale.
    This is a decision aid, not an algorithm — the output informs judgement.
    """
    if effort_weeks <= 0:
        return ("DEFER_ASSESS", "Effort unknown — do not absorb until scoped.")
    if effort_weeks > phase_capacity_remaining:
        return ("NO_PHASE", f"Effort {effort_weeks}w exceeds remaining capacity {phase_capacity_remaining:.1f}w.")
    if is_in_scope_extension:
        return ("YES_TRADE", "Fits existing story scope; displaces an equal-cost Could item.")
    return ("NO_PHASE", "New scope — valid but not this phase; add to next-phase backlog.")

```
classify_change("watch history tab on billing screen", True,  0.5, 2.0)
→ ("YES_TRADE", "Fits existing story scope; displaces an equal-cost Could item.")

classify_change("founder MRR dashboard",               False, 2.0, 2.0)
→ ("NO_PHASE", "New scope — valid but not this phase; add to next-phase backlog.")

classify_change("real-time revenue alerting",           False, 0,   2.0)
→ ("DEFER_ASSESS", "Effort unknown — do not absorb until scoped.")
```

### 2.2 The scope trade

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class ScopeItem:
    name: str
    moscow: str     # Must / Should / Could / Won't
    effort_weeks: float
    status: str = "committed"  # committed / candidate_for_removal

@dataclass
class ScopeBasket:
    capacity_weeks: float
    items: List[ScopeItem] = field(default_factory=list)

    def total_committed(self):
        return sum(i.effort_weeks for i in self.items if i.status == "committed")

    def remaining(self):
        return self.capacity_weeks - self.total_committed()

    def can_add(self, effort_weeks):
        return self.remaining() >= effort_weeks

    def add_requires_trade(self, new_item, displaced_item=None):
        """Returns the trade decision."""
        if self.can_add(new_item.effort_weeks):
            return ("YES_DIRECT", f"Capacity has room ({self.remaining():.1f}w remaining).")
        if displaced_item:
            displaced_item.status = "candidate_for_removal"
            return ("YES_TRADE", f"Trade: add {new_item.name!r} ({new_item.effort_weeks}w), "
                                 f"remove {displaced_item.name!r} ({displaced_item.effort_weeks}w).")
        return ("NO_ROOM", f"No room without a trade. Remaining: {self.remaining():.1f}w.")

In [ ]:
basket = ScopeBasket(capacity_weeks=6.0, items=[
    ScopeItem("Support billing lookup",   "Must",   1.0),
    ScopeItem("New-hire refund wiki",     "Should", 1.5),
    ScopeItem("Support complaint themes", "Should", 1.5),
    ScopeItem("Content catalog count",    "Could",  0.5),
])
# Total committed: 4.5w out of 6.0w = 1.5w remaining

extension = ScopeItem("Watch history tab",  "Could", 0.5)
basket.add_requires_trade(extension)
# → ("YES_DIRECT", "Capacity has room (1.5w remaining).")

new_scope = ScopeItem("Founder MRR dashboard", "Should", 2.0)
catalog   = ScopeItem("Content catalog count",  "Could",  0.5)
basket.add_requires_trade(new_scope, displaced_item=catalog)
# → ("YES_TRADE", "Trade: add 'Founder MRR dashboard' (2.0w), remove 'Content catalog count' (0.5w).")
# But this only covers 0.5w of a 2.0w add — 1.5w of displacement still needed.

### 2.3 MVP discipline: the acceptance criteria line

In [ ]:
def check_mvp_scope(story_name, acceptance_criteria_met, proposed_additions):
    """
    Determines whether proposed additions are in-MVP or above-MVP.
    acceptance_criteria_met: dict of criterion -> bool
    proposed_additions: list of feature description strings
    """
    unmet = [k for k, v in acceptance_criteria_met.items() if not v]
    if unmet:
        return {
            "mvp_status": "INCOMPLETE",
            "unmet_criteria": unmet,
            "verdict": "Finish the MVP before considering additions.",
            "additions": "ALL deferred until MVP criteria are met.",
        }
    return {
        "mvp_status": "COMPLETE",
        "unmet_criteria": [],
        "verdict": f"{story_name} MVP is done. Additions are new scope decisions.",
        "additions": {a: "New scope — assess via change request process" for a in proposed_additions},
    }

In [ ]:
# Story 1 MVP — billing lookup — vs watch history addition
check_mvp_scope(
    "Story 1: Support billing lookup",
    {
        "email exists → returns last-60-day charge list": True,
        "answer reads from live DB, not static report": True,
        "support agent can query without engineering escalation": True,
    },
    proposed_additions=["watch history tab alongside charges"]
)
# →
# {
#   "mvp_status": "COMPLETE",
#   "verdict": "Story 1 MVP is done. Additions are new scope decisions.",
#   "additions": {"watch history tab alongside charges": "New scope — assess via change request process"}
# }

### 2.4 Saying no: language that protects the relationship

In [ ]:
def draft_response(change_request, bucket, future_home=None, trade_offer=None):
    templates = {
        "YES_TRADE": (
            f"Let's add '{change_request}' — it fits the current arc. To make room, "
            f"we'd need to move {trade_offer!r} to Could or drop it this phase. "
            f"Does that trade work for you?"
        ),
        "NO_PHASE": (
            f"'{change_request}' is worth building — I don't want to lose it. "
            f"This phase it would displace work we've both committed to, so I'd put it "
            f"in '{future_home}' as the first candidate for next engagement. "
            f"Is there a story in the current scope you'd trade for it today?"
        ),
        "DEFER_ASSESS": (
            f"'{change_request}' sounds valuable — I need one working session to scope "
            f"the effort before I can say yes or no. Can I come back to you by end of week "
            f"with a size and a trade-off?"
        ),
    }
    return templates[bucket]

```
draft_response("Founder MRR dashboard", "NO_PHASE", future_home="Phase 2: founder tools")
→
"'Founder MRR dashboard' is worth building — I don't want to lose it. This phase it would
 displace work we've both committed to, so I'd put it in 'Phase 2: founder tools' as the
 first candidate for next engagement. Is there a story in the current scope you'd trade
 for it today?"
```

## 3. CinemaStream in Practice — On Loan at FilmiBox

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class ChangeRequest:
    description: str
    source: str  # who asked
    effort_estimate_weeks: float
    is_extension_of_story: Optional[str] = None  # None if new scope

@dataclass
class ActiveScope:
    capacity_weeks: float
    phase_weeks_remaining: float
    items: list

    def assess(self, cr: ChangeRequest):
        if cr.effort_estimate_weeks <= 0:
            return ("DEFER_ASSESS", "Scope unknown. Need sizing before deciding.")
        if cr.effort_estimate_weeks > self.phase_weeks_remaining:
            return ("NO_PHASE",
                    f"Effort {cr.effort_estimate_weeks}w exceeds {self.phase_weeks_remaining}w remaining.")
        if cr.is_extension_of_story:
            return ("YES_TRADE",
                    f"Extension of {cr.is_extension_of_story}. Absorbs {cr.effort_estimate_weeks}w "
                    f"from remaining capacity ({self.phase_weeks_remaining}w).")
        return ("NO_PHASE", "New scope. Valid, but would displace committed work.")

scope = ActiveScope(
    capacity_weeks=6.0,
    phase_weeks_remaining=2.2,  # 3.8w spent, 2.2w left
    items=["Story 1 (done)", "Story 4 wiki (Should)", "Story 2 themes (Should)", "Story 3 catalog (Could)"],
)

watch_history_cr = ChangeRequest(
    description="Watch history tab alongside billing charges",
    source="Support team lead",
    effort_estimate_weeks=0.5,
    is_extension_of_story="Story 1: Support billing lookup",
)

mrr_dashboard_cr = ChangeRequest(
    description="Founder churn-adjusted MRR dashboard",
    source="Co-founder (Dev)",
    effort_estimate_weeks=2.0,
    is_extension_of_story=None,
)

print("CR 1:", scope.assess(watch_history_cr))
print("CR 2:", scope.assess(mrr_dashboard_cr))

```
CR 1: ('YES_TRADE', 'Extension of Story 1: Support billing lookup. Absorbs 0.5w from remaining capacity (2.2w).')
CR 2: ('NO_PHASE', 'New scope. Valid, but would displace committed work.')
```

In [ ]:
# Final updated scope after both CRs assessed
SCOPE_V2 = {
    "Must":  ["1. Support billing lookup + watch history (extended, 1.5w)"],
    "Should":["4. New-hire refund wiki (1.5w)", "2. Support complaint themes (1.5w)"],
    "Could": [],  # catalog count dropped to fund CR 1
    "Won't": [
        "5. Founder churn-adj MRR (this phase) — Phase 2, founders-only separate system",
        "3. Content catalog count (this phase) — traded for watch-history extension",
    ],
}

for bucket, items in SCOPE_V2.items():
    print(f"\n{bucket}:")
    for i in items:
        print(f"  • {i}")

```
Must:
  • 1. Support billing lookup + watch history (extended, 1.5w)

Should:
  • 4. New-hire refund wiki (1.5w)
  • 2. Support complaint themes (1.5w)

Could:

Won't:
  • 5. Founder churn-adj MRR (this phase) — Phase 2, founders-only separate system
  • 3. Content catalog count (this phase) — traded for watch-history extension
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 90: The PoV-PoC-MVP Lifecycle — Proving Value Before Proving Feasibility Before Earning Adoption

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 Three stages, three risks

In [ ]:
# The lifecycle as data: each stage retires exactly one kind of risk,
# answers exactly one question, and produces exactly one deliverable.
LIFECYCLE = [
    ("PoV", "Proof of Value", "Value risk",
     "Is this worth building at all?",
     "A written hypothesis with measurable success criteria"),
    ("PoC", "Proof of Concept", "Feasibility risk",
     "Can we build it with this data and stack?",
     "Working code, time-boxed, deliberately ugly"),
    ("MVP", "Minimum Viable Product", "Adoption risk",
     "Will real users actually use it?",
     "The smallest version that satisfies the acceptance criteria"),
]

for code, name, risk, question, deliverable in LIFECYCLE:
    print(f"{code} - {name}")
    print(f"   Retires:     {risk}")
    print(f"   Question:    {question}")
    print(f"   Deliverable: {deliverable}")
    print()

```
PoV - Proof of Value
   Retires:     Value risk
   Question:    Is this worth building at all?
   Deliverable: A written hypothesis with measurable success criteria

PoC - Proof of Concept
   Retires:     Feasibility risk
   Question:    Can we build it with this data and stack?
   Deliverable: Working code, time-boxed, deliberately ugly

MVP - Minimum Viable Product
   Retires:     Adoption risk
   Question:    Will real users actually use it?
   Deliverable: The smallest version that satisfies the acceptance criteria
```

### 2.2 PoV articulation: the hypothesis you can lose

In [ ]:
from dataclasses import dataclass

@dataclass
class PoV:
    """A Proof of Value: a falsifiable bet on business value.

    Every field is load-bearing. A PoV missing any of these
    is an opinion, not a hypothesis.
    """
    hypothesis: str        # what we believe will happen if this exists
    metric: str            # the ONE number that proves or disproves it
    baseline: str          # where that number is today (measured, not guessed)
    target: str            # where it must reach for the bet to pay off
    time_box_weeks: float  # how long we give ourselves to find out

    def is_testable(self):
        """Checks the PoV can actually be won or lost."""
        problems = []
        if not self.metric.strip():
            problems.append("no metric - 'better' is not measurable")
        if not self.baseline.strip():
            problems.append("no baseline - you cannot show improvement without a starting point")
        if not self.target.strip():
            problems.append("no target - when do you declare success?")
        if self.time_box_weeks <= 0 or self.time_box_weeks > 4:
            problems.append("time box missing or too long - a PoV is weeks, not months")
        return (len(problems) == 0, problems)

# A PoV the way clients first say it:
vague = PoV(
    hypothesis="AI will make our support team better",
    metric="", baseline="", target="", time_box_weeks=0,
)
ok, problems = vague.is_testable()
print(f"Testable: {ok}")
for p in problems:
    print(f"  - {p}")

```
Testable: False
  - no metric - 'better' is not measurable
  - no baseline - you cannot show improvement without a starting point
  - no target - when do you declare success?
  - time box missing or too long - a PoV is weeks, not months
```

In [ ]:
from dataclasses import dataclass

@dataclass
class PoV:
    hypothesis: str
    metric: str
    baseline: str
    target: str
    time_box_weeks: float

    def is_testable(self):
        problems = []
        if not self.metric.strip():
            problems.append("no metric - 'better' is not measurable")
        if not self.baseline.strip():
            problems.append("no baseline - you cannot show improvement without a starting point")
        if not self.target.strip():
            problems.append("no target - when do you declare success?")
        if self.time_box_weeks <= 0 or self.time_box_weeks > 4:
            problems.append("time box missing or too long - a PoV is weeks, not months")
        return (len(problems) == 0, problems)

sharp = PoV(
    hypothesis="If support agents can look up charges themselves, "
               "billing questions stop escalating to engineering",
    metric="% of billing queries resolved without engineering escalation",
    baseline="0% - today every billing question becomes an engineering ticket",
    target=">=70% resolved in-tool during a one-week pilot",
    time_box_weeks=2.0,
)
ok, problems = sharp.is_testable()
print(f"Testable: {ok}")
print(f"The bet: {sharp.hypothesis}")
print(f"Wins if: {sharp.metric} goes from [{sharp.baseline}] to [{sharp.target}]")
print(f"We know within: {sharp.time_box_weeks} weeks")

```
Testable: True
The bet: If support agents can look up charges themselves, billing questions stop escalating to engineering
Wins if: % of billing queries resolved without engineering escalation goes from [0% - today every billing question becomes an engineering ticket] to [>=70% resolved in-tool during a one-week pilot]
We know within: 2.0 weeks
```

### 2.3 PoC scoping: time-boxed, ugly on purpose

In [ ]:
def review_poc(criteria_results, pivot_insight=None):
    """The PoC gate: PROCEED, PIVOT, or KILL.

    criteria_results: dict of success criterion -> bool (met or not),
                      written BEFORE the PoC started, not after.
    pivot_insight:    if the PoC missed but revealed a better direction,
                      name it here - that converts a KILL into a PIVOT.
    """
    met = sum(1 for v in criteria_results.values() if v)
    total = len(criteria_results)
    if met == total:
        return ("PROCEED", f"All {total} criteria met -> fund the MVP.")
    if pivot_insight:
        return ("PIVOT", f"{met}/{total} criteria met, but the PoC revealed: {pivot_insight}")
    return ("KILL", f"Only {met}/{total} criteria met and no new direction -> stop. "
                    "Two weeks spent learning this is cheap. Three months would not be.")

# Three PoCs, three outcomes:
all_met = {"answers from live data": True, "under 5s response": True, "handles real schema": True}
print(review_poc(all_met))

partial = {"answers from live data": True, "under 5s response": False, "handles real schema": False}
print(review_poc(partial, pivot_insight="the live API is too slow, but a nightly "
                                        "extract supports 90% of the queries"))

print(review_poc(partial))

```
('PROCEED', 'All 3 criteria met -> fund the MVP.')
('PIVOT', '1/3 criteria met, but the PoC revealed: the live API is too slow, but a nightly extract supports 90% of the queries')
('KILL', 'Only 1/3 criteria met and no new direction -> stop. Two weeks spent learning this is cheap. Three months would not be.')
```

### 2.4 MVP definition: the acceptance criteria line, plus the production floor

In [ ]:
# What upgrades when a PoC graduates to MVP.
# The feature is the same. Everything around it changes.
POC_TO_MVP = {
    "hardcoded credentials":      "real auth + secrets management",
    "happy path only":            "error handling for the failures users will actually hit",
    "manually run by the FDE":    "deployed where users reach it themselves",
    "prints to the console":      "logging someone can read at 2am",
    "works on the demo extract":  "works on the live, messy, current data",
    "no one is on the hook":      "a named owner and a runbook entry",
}

print("PoC                            ->  MVP")
print("-" * 75)
for poc_state, mvp_state in POC_TO_MVP.items():
    print(f"{poc_state:<30} ->  {mvp_state}")

```
PoC                            ->  MVP
---------------------------------------------------------------------------
hardcoded credentials          ->  real auth + secrets management
happy path only                ->  error handling for the failures users will actually hit
manually run by the FDE        ->  deployed where users reach it themselves
prints to the console          ->  logging someone can read at 2am
works on the demo extract      ->  works on the live, messy, current data
no one is on the hook          ->  a named owner and a runbook entry
```

## 3. CinemaStream in Practice — On Loan at FilmiBox

In [ ]:
from dataclasses import dataclass

@dataclass
class PoV:
    hypothesis: str
    metric: str
    baseline: str
    target: str
    time_box_weeks: float

    def is_testable(self):
        problems = []
        if not self.metric.strip():
            problems.append("no metric")
        if not self.baseline.strip():
            problems.append("no baseline")
        if not self.target.strip():
            problems.append("no target")
        if self.time_box_weeks <= 0 or self.time_box_weeks > 4:
            problems.append("bad time box")
        return (len(problems) == 0, problems)

def review_poc(criteria_results, pivot_insight=None):
    met = sum(1 for v in criteria_results.values() if v)
    total = len(criteria_results)
    if met == total:
        return ("PROCEED", f"All {total} criteria met -> fund the MVP.")
    if pivot_insight:
        return ("PIVOT", f"{met}/{total} criteria met, but the PoC revealed: {pivot_insight}")
    return ("KILL", f"Only {met}/{total} criteria met and no new direction -> stop.")

# Story 1's PoV - written in discovery (Ch 086-087), formalized here:
story1_pov = PoV(
    hypothesis="If support agents can look up a subscriber's charges themselves, "
               "billing questions stop escalating to engineering",
    metric="% of billing queries resolved in-tool without engineering escalation",
    baseline="0% - every billing question becomes an engineering ticket "
             "(~4h round-trip)",
    target=">=70% resolved in-tool during a one-week pilot with 5 agents",
    time_box_weeks=2.0,
)
ok, _ = story1_pov.is_testable()
print(f"Story 1 PoV testable: {ok}")

# The week-4 pilot numbers: 5 agents, one week, 14 billing queries.
queries_total = 14
resolved_in_tool = 11   # the other 3: two needed a refund (out of scope), one was a bank-side issue
pilot_results = {
    ">=70% of billing queries resolved in-tool without escalation":
        resolved_in_tool / queries_total >= 0.70,
    "median time-to-answer under 5 minutes (baseline: ~4h escalation round-trip)":
        True,   # pilot median was ~3 minutes
    "answers come from the live read replica, not a static extract":
        True,
}
print(f"Pilot: {resolved_in_tool}/{queries_total} resolved in-tool "
      f"({resolved_in_tool / queries_total:.1%})")
print(f"Gate: {review_poc(pilot_results)}")

```
Story 1 PoV testable: True
Pilot: 11/14 resolved in-tool (78.6%)
Gate: ('PROCEED', 'All 3 criteria met -> fund the MVP.')
```

In [ ]:
from dataclasses import dataclass

@dataclass
class PoV:
    hypothesis: str
    metric: str
    baseline: str
    target: str
    time_box_weeks: float

    def is_testable(self):
        problems = []
        if not self.metric.strip():
            problems.append("no metric")
        if not self.baseline.strip():
            problems.append("no baseline")
        if not self.target.strip():
            problems.append("no target")
        if self.time_box_weeks <= 0 or self.time_box_weeks > 4:
            problems.append("bad time box")
        return (len(problems) == 0, problems)

# Phase 2, first item (per the Ch 089 deferral): the founders-only MRR view.
phase2_pov = PoV(
    hypothesis="A founders-only churn-adjusted MRR view replaces the monthly "
               "spreadsheet assembly and surfaces churn-driven revenue dips "
               "weeks earlier",
    metric="time from 'Anjali wants the number' to having it, and the lag "
           "between a churn-driven dip and a founder seeing it",
    baseline="~4 hours of manual assembly, once a month - a dip can sit "
             "invisible for up to 30 days",
    target="on-demand in under a minute, refreshed daily - dips visible "
           "within 24 hours",
    time_box_weeks=2.0,
)
ok, _ = phase2_pov.is_testable()
print(f"Phase 2 PoV testable: {ok}")
print(f"\nThe bet:  {phase2_pov.hypothesis}")
print(f"Metric:   {phase2_pov.metric}")
print(f"Baseline: {phase2_pov.baseline}")
print(f"Target:   {phase2_pov.target}")
print(f"Time box: {phase2_pov.time_box_weeks} weeks (matches the CR2 estimate from week 3)")

```
Phase 2 PoV testable: True

The bet:  A founders-only churn-adjusted MRR view replaces the monthly spreadsheet assembly and surfaces churn-driven revenue dips weeks earlier
Metric:   time from 'Anjali wants the number' to having it, and the lag between a churn-driven dip and a founder seeing it
Baseline: ~4 hours of manual assembly, once a month - a dip can sit invisible for up to 30 days
Target:   on-demand in under a minute, refreshed daily - dips visible within 24 hours
Time box: 2.0 weeks (matches the CR2 estimate from week 3)
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 91: Demo Storytelling — Making Someone Who Wasn't in the Room Feel It

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 The three-act demo structure

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class DemoAct:
    act: str          # "before" | "change" | "after"
    headline: str     # one-sentence summary
    script: str       # what you say (or show)
    duration_sec: int

@dataclass
class DemoStory:
    story_name: str
    hero: str             # the person whose day changes
    pain: str             # what was painful before
    acts: List[DemoAct]

    def total_duration_min(self):
        return sum(a.duration_sec for a in self.acts) / 60

    def check_structure(self):
        act_names = [a.act for a in self.acts]
        required = {"before", "change", "after"}
        missing = required - set(act_names)
        if missing:
            return f"INCOMPLETE — missing acts: {missing}"
        before = next(a for a in self.acts if a.act == "before")
        change = next(a for a in self.acts if a.act == "change")
        if change.duration_sec > before.duration_sec * 2:
            return "WARNING — change act is longer than twice the before act. Demo may feel like a tour."
        return "OK"

```
billing_story = DemoStory(
    story_name="Support billing lookup",
    hero="Support agent",
    pain="Subscriber waits 4+ hours for a yes/no answer about a charge",
    acts=[
        DemoAct("before",  "A 24-hour wait for a two-word answer", "...", 90),
        DemoAct("change",  "Live lookup: email → charges in 30s",  "...", 60),
        DemoAct("after",   "Same query resolved before the call ends", "...", 45),
    ]
)
billing_story.check_structure()  → "OK"
billing_story.total_duration_min()  → 3.25
```

### 2.2 The hero mapping

In [ ]:
def map_hero(story_name, tool_description, user_role, before_state, after_state):
    print(f"Story:   {story_name}")
    print(f"Tool:    {tool_description}  ← this is NOT the hero")
    print(f"Hero:    {user_role}")
    print(f"Before:  {before_state}")
    print(f"After:   {after_state}")
    print()

map_hero(
    "Support billing lookup",
    "MCP tool querying subscriber DB",
    "Support agent on the phone with an upset subscriber",
    "Opens a ticket. Waits for engineering. Calls back. Sometimes next day.",
    "Answers the question before the subscriber hangs up. No ticket opened.",
)

map_hero(
    "New-hire refund wiki",
    "Bilingual RAG over company knowledge base",
    "New hire on first solo shift, unsure about the refund process",
    "Interrupts a senior colleague. Feels embarrassed. Slows the team.",
    "Asks the wiki in Hindi. Gets the answer with a source link. Nobody interrupted.",
)

```
Story:   Support billing lookup
Tool:    MCP tool querying subscriber DB  ← this is NOT the hero
Hero:    Support agent on the phone with an upset subscriber
Before:  Opens a ticket. Waits for engineering. Calls back. Sometimes next day.
After:   Answers the question before the subscriber hangs up. No ticket opened.

Story:   New-hire refund wiki
Tool:    Bilingual RAG over company knowledge base
Hero:    New hire on first solo shift, unsure about the refund process
Before:  Interrupts a senior colleague. Feels embarrassed. Slows the team.
After:   Asks the wiki in Hindi. Gets the answer with a source link. Nobody interrupted.
```

### 2.3 The live demo risk — and the fallback

In [ ]:
DEMO_PREP_CHECKLIST = [
    ("LIVE_DATA",      "Verify the data source is reachable from the demo device"),
    ("CREDENTIALS",    "Confirm API keys and DB credentials have not rotated since last run"),
    ("FALLBACK",       "Record a screen-capture fallback video on the morning of the demo"),
    ("TIMER",          "Rehearse the change act — it should feel unhurried in ≤90s"),
    ("SILENT_WINDOW",  "Leave the demo interface open before the meeting — no live typing of URLs"),
    ("QUESTION_PREP",  "List the 3 questions most likely to come, with one-sentence answers"),
    ("CLOSE",          "End with 'What questions do you have?' not 'So that's the demo.'"),
]

def print_checklist():
    print("Demo prep checklist:")
    for i, (label, action) in enumerate(DEMO_PREP_CHECKLIST, 1):
        print(f"  {i}. [{label:16s}] {action}")

```
Demo prep checklist:
  1. [LIVE_DATA        ] Verify the data source is reachable from the demo device
  2. [CREDENTIALS      ] Confirm API keys and DB credentials have not rotated since last run
  3. [FALLBACK         ] Record a screen-capture fallback video on the morning of the demo
  4. [TIMER            ] Rehearse the change act — it should feel unhurried in ≤90s
  5. [SILENT_WINDOW    ] Leave the demo interface open before the meeting — no live typing of URLs
  6. [QUESTION_PREP    ] List the 3 questions most likely to come, with one-sentence answers
  7. [CLOSE            ] End with 'What questions do you have?' not 'So that's the demo.'
```

### 2.4 The question most engineers don't rehearse

In [ ]:
def draft_before_state(user_role, specific_scenario, time_cost, emotional_cost):
    """
    Before state should be specific (a named scenario), 
    have a concrete cost (time or money), 
    and name the emotional reality (frustration, embarrassment, uncertainty).
    """
    return (
        f"Picture {user_role}. "
        f"{specific_scenario}. "
        f"That {time_cost}. "
        f"{emotional_cost}."
    )

result = draft_before_state(
    user_role="your newest hire on their first solo shift",
    specific_scenario="A subscriber calls about a refund procedure they haven't seen before. "
                      "They can't find it in the shared drive. They interrupt a senior colleague",
    time_cost="costs your senior staff fifteen minutes they weren't planning to spend",
    emotional_cost="And the new hire starts their day feeling like a burden",
)
print(result)

```
Picture your newest hire on their first solo shift. A subscriber calls about a refund
procedure they haven't seen before. They can't find it in the shared drive. They interrupt
a senior colleague. That costs your senior staff fifteen minutes they weren't planning to
spend. And the new hire starts their day feeling like a burden.
```

## 3. CinemaStream in Practice — FilmiBox Demo Day

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class Act:
    kind: str; headline: str; script: str; seconds: int

@dataclass
class FilmiBoxDemo:
    story: str; hero: str; acts: List[Act]
    pilot_result: str = ""

    def duration_min(self):
        return round(sum(a.seconds for a in self.acts) / 60, 1)

    def print_runsheet(self):
        print(f"\n{'='*60}")
        print(f"Story: {self.story}")
        print(f"Hero:  {self.hero}")
        if self.pilot_result:
            print(f"Data:  {self.pilot_result}")
        for act in self.acts:
            bar = "▶" if act.kind == "change" else " "
            print(f"  {bar} [{act.kind.upper():6s} {act.seconds:3d}s] {act.headline}")
        print(f"  Total: {self.duration_min()} min")


BILLING = FilmiBoxDemo(
    story="1. Support billing lookup + watch history",
    hero="Support agent on the phone with an upset subscriber",
    pilot_result="11 of 14 queries resolved in-tool (78.6%) vs ≥70% target — PROCEED",
    acts=[
        Act("before", "A subscriber waits 4 hours for a yes/no",
            "Picture your support agent on a call. A subscriber says 'I was charged twice.' "
            "They need a yes or a no. Before today, the agent had to open an engineering ticket, "
            "wait — sometimes hours, sometimes until tomorrow — and call back. The subscriber is "
            "still on hold. Let me show you what it looks like now.",
            90),
        Act("change", "Live: email → charges + watch history in 30 seconds",
            "[TYPE EMAIL. SHOW CHARGES. SHOW LAST 5 WATCH EVENTS.] "
            "Thirty seconds. No ticket. No paging engineering. The subscriber is still on the line.",
            60),
        Act("after", "Same question resolved before the call ends",
            "That subscriber's question — 'was I charged twice?' — answered before they hung up. "
            "In the pilot this week, 11 of 14 billing queries resolved this way. "
            "The other 3 were refund edge cases that are genuinely outside this tool's scope — "
            "they escalated appropriately. The ones that escalated now do so with a reason, "
            "not just 'I don't know yet.'",
            60),
    ]
)

WIKI = FilmiBoxDemo(
    story="4. New-hire refund wiki (bilingual)",
    hero="New hire on first solo shift",
    acts=[
        Act("before", "A new hire's first solo call ends with an interruption",
            "Picture your newest hire on their first solo day. A subscriber asks about your "
            "refund policy for a cancelled subscription. They look in the shared drive — "
            "three folders, six documents, nothing clear. They interrupt a senior colleague. "
            "Fifteen minutes of someone else's morning. The new hire feels like a burden "
            "before they've finished their first shift.",
            75),
        Act("change", "Live: 'how do I issue a refund?' in Hindi → answer in 8 seconds",
            "[TYPE 'refund kaise karein' IN HINDI. SHOW ANSWER WITH SOURCE LINK.]",
            45),
        Act("after", "New hire self-serves, senior staff uninterrupted",
            "Hindi or English. The answer comes with a source link so they can read the full "
            "policy if they need to. Your senior staff's morning is their own. "
            "And the new hire learns where the answer lives — next time they look there first.",
            50),
    ]
)

THEMES = FilmiBoxDemo(
    story="2. Support complaint themes — early bug signal",
    hero="Support lead skimming 400 tickets a week",
    acts=[
        Act("before", "A bug hides in 400 tickets for a week",
            "Your support lead reads roughly 400 tickets a week to find patterns. "
            "Last month a subtitle bug was reported by 30 subscribers over 4 days before anyone "
            "connected the dots — by then there were 90 open tickets. "
            "That is the problem: patterns are in the data before a human can see them.",
            70),
        Act("change", "Live: top 5 complaint themes from last 7 days",
            "[RUN THEME QUERY. SHOW TOP 5 THEMES WITH TICKET COUNTS.]",
            50),
        Act("after", "Patterns visible in hours, not days",
            "That query runs in seconds. If a bug starts at 10am, by the afternoon stand-up "
            "it is a named theme, not a noise spike. Your support lead still reads the tickets "
            "that need a human — but they read *toward* something, not hoping to spot it.",
            55),
    ]
)

def run_demo_day():
    demos = [BILLING, WIKI, THEMES]
    print("FilmiBox Demo Day — Week 5 Runsheet")
    print(f"Stories: {len(demos)}  |  "
          f"Total: {sum(d.duration_min() for d in demos):.1f} min + Q&A")

    for d in demos:
        d.print_runsheet()

    print("\n" + "="*60)
    print("Prep checklist (verify before the call):")
    checks = [
        "DB live: subscriber_db reachable from demo machine",
        "Wiki RAG index fresh (last ingested <24h ago)",
        "Zendesk tickets indexed through yesterday",
        "Screen-capture fallback recorded this morning",
        "Billing demo rehearsed — change act under 90s",
        "Three likely questions prepared: Phase 2, Hindi languages, ticket security",
    ]
    for i, c in enumerate(checks, 1):
        print(f"  {i}. {c}")

run_demo_day()

```
FilmiBox Demo Day — Week 5 Runsheet
Stories: 3  |  Total: 19.5 min + Q&A

============================================================
Story: 1. Support billing lookup + watch history
Hero:  Support agent on the phone with an upset subscriber
Data:  11 of 14 queries resolved in-tool (78.6%) vs ≥70% target — PROCEED
   [BEFORE  90s] A subscriber waits 4 hours for a yes/no
 ▶ [CHANGE  60s] Live: email → charges + watch history in 30 seconds
   [AFTER   60s] Same question resolved before the call ends
  Total: 3.5 min

============================================================
Story: 4. New-hire refund wiki (bilingual)
Hero:  New hire on first solo shift
   [BEFORE  75s] A new hire's first solo call ends with an interruption
 ▶ [CHANGE  45s] Live: 'how do I issue a refund?' in Hindi → answer in 8 seconds
   [AFTER   50s] New hire self-serves, senior staff uninterrupted
  Total: 2.8 min

============================================================
Story: 2. Support complaint themes — early bug signal
Hero:  Support lead skimming 400 tickets a week
   [BEFORE  70s] A bug hides in 400 tickets for a week
 ▶ [CHANGE  50s] Live: top 5 complaint themes from last 7 days
   [AFTER   55s] Patterns visible in hours, not days
  Total: 2.9 min

============================================================
Prep checklist (verify before the call):
  1. DB live: subscriber_db reachable from demo machine
  2. Wiki RAG index fresh (last ingested <24h ago)
  3. Zendesk tickets indexed through yesterday
  4. Screen-capture fallback recorded this morning
  5. Billing demo rehearsed — change act under 90s
  6. Three likely questions prepared: Phase 2, Hindi languages, ticket security
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 92: Adoption Strategy — Making What You Built Actually Get Used

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 Training design: skill transfer, not feature walkthrough

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class TrainingScenario:
    scenario_name: str
    role: str               # who this is designed for
    setup: str              # the situation the trainee is in
    steps: List[str]        # what they do, step by step
    common_failure: str     # what goes wrong if they skip a step
    success_signal: str     # how they know it worked

@dataclass
class TrainingPlan:
    tool_name: str
    audience: str
    duration_minutes: int
    scenarios: List[TrainingScenario]
    self_practice_task: str     # the "now you do one" closing task
    owner_contact: str          # who they call if something breaks

def format_training_plan(plan: TrainingPlan) -> str:
    lines = [
        f"Training: {plan.tool_name}",
        f"Audience: {plan.audience}",
        f"Duration: {plan.duration_minutes} min",
        f"Owner contact: {plan.owner_contact}",
        "",
        "Scenarios:",
    ]
    for i, s in enumerate(plan.scenarios, 1):
        lines.append(f"  {i}. [{s.role}] {s.scenario_name}")
        lines.append(f"     Setup: {s.setup}")
        for j, step in enumerate(s.steps, 1):
            lines.append(f"     Step {j}: {step}")
        lines.append(f"     Watch for: {s.common_failure}")
        lines.append(f"     Done when: {s.success_signal}")
        lines.append("")
    lines.append(f"Closing task (trainee solo): {plan.self_practice_task}")
    return "\n".join(lines)

### 2.2 Success metrics: adoption is measurable

In [ ]:
@dataclass
class AdoptionMetric:
    metric_name: str
    definition: str         # exactly how it is measured
    baseline: float         # where you are on launch day
    target: float           # where you want to be by the target date
    target_date: str        # when you expect to hit it
    current: float = 0.0

@dataclass
class AdoptionPlan:
    tool_name: str
    launch_date: str
    metrics: List[AdoptionMetric]
    review_cadence: str     # how often you check

def check_adoption_status(plan: AdoptionPlan) -> List[str]:
    report = [f"Adoption status: {plan.tool_name}", f"Review cadence: {plan.review_cadence}", ""]
    for m in plan.metrics:
        if m.target == m.baseline:
            pct = 100.0
        else:
            pct = min(100.0, (m.current - m.baseline) / (m.target - m.baseline) * 100)
        status = "✓ ON TRACK" if pct >= 70 else "⚠ AT RISK" if pct >= 40 else "✗ STALLED"
        report.append(f"{status}  {m.metric_name}")
        report.append(f"          baseline={m.baseline} | current={m.current} | target={m.target} by {m.target_date}")
        report.append(f"          {m.definition}")
        report.append("")
    return report

# Example: define adoption metrics for a tool
metrics = [
    AdoptionMetric(
        metric_name="Weekly active users",
        definition="Distinct users who submitted ≥1 query in the past 7 days",
        baseline=0.0,
        target=0.80,  # 80% of 5 support agents = 4 agents
        target_date="2026-07-15",
        current=0.60,
    ),
    AdoptionMetric(
        metric_name="In-tool resolution rate",
        definition="Queries resolved in tool / total queries (no escalation)",
        baseline=0.0,
        target=0.70,
        target_date="2026-07-15",
        current=0.786,
    ),
    AdoptionMetric(
        metric_name="Manual ticket rate for billing queries",
        definition="Billing tickets created per week / total billing contacts",
        baseline=1.0,   # 100% of queries went to tickets before launch
        target=0.30,    # target: 70% deflected
        target_date="2026-07-15",
        current=0.50,
    ),
]

plan = AdoptionPlan(
    tool_name="Billing Lookup Tool",
    launch_date="2026-07-01",
    metrics=metrics,
    review_cadence="Weekly for first 30 days, biweekly after",
)

for line in check_adoption_status(plan):
    print(line)

### 2.3 The handover document: written for eighteen months from now

In [ ]:
@dataclass
class HandoverSection:
    title: str
    content: str

@dataclass
class HandoverDocument:
    system_name: str
    engagement_end_date: str
    internal_owner: str
    escalation_contact: str   # the FDE, for the first 90 days
    sections: List[HandoverSection]

def format_handover(doc: HandoverDocument) -> str:
    lines = [
        f"# Handover: {doc.system_name}",
        f"Engagement ended: {doc.engagement_end_date}",
        f"Internal owner: {doc.internal_owner}",
        f"Escalation (first 90 days): {doc.escalation_contact}",
        "",
    ]
    for s in doc.sections:
        lines.append(f"## {s.title}")
        lines.append(s.content)
        lines.append("")
    return "\n".join(lines)

STANDARD_SECTIONS = [
    "What this system does (one paragraph, non-technical)",
    "What it does NOT do (scope boundaries, explicitly stated)",
    "How to access it (URL/CLI/env, credentials location)",
    "How to check if it is healthy (what green looks like)",
    "How to restart it if it breaks (step by step)",
    "How to update the knowledge base / index (step by step)",
    "What to do if it gives a wrong answer (escalation path)",
    "Who owns it and who to call (internal + FDE contact for 90 days)",
    "Known limitations (what the system is bad at, documented honestly)",
    "Success metrics and where to find them (dashboard / query)",
]

```
Output (STANDARD_SECTIONS):
['What this system does (one paragraph, non-technical)',
 'What it does NOT do (scope boundaries, explicitly stated)',
 'How to access it (URL/CLI/env, credentials location)',
 'How to check if it is healthy (what green looks like)',
 'How to restart it if it breaks (step by step)',
 'How to update the knowledge base / index (step by step)',
 'What to do if it gives a wrong answer (escalation path)',
 'Who owns it and who to call (internal + FDE contact for 90 days)',
 'Known limitations (what the system is bad at, documented honestly)',
 'Success metrics and where to find them (dashboard / query)']
```

### 2.4 The handover checklist: a gate, not a courtesy

In [ ]:
@dataclass
class HandoverChecklistItem:
    item: str
    verified_by: str    # who signs off
    done: bool = False

def run_handover_checklist(items: List[HandoverChecklistItem]) -> None:
    total = len(items)
    done = sum(1 for i in items if i.done)
    print(f"Handover checklist: {done}/{total} complete")
    print()
    for item in items:
        status = "✓" if item.done else "✗"
        print(f"  [{status}] {item.item}")
        print(f"       Verified by: {item.verified_by}")

STANDARD_CHECKLIST = [
    HandoverChecklistItem("Internal owner can log in and submit a query", "internal owner demo", done=True),
    HandoverChecklistItem("Internal owner can check system health (index freshness, uptime)", "internal owner demo", done=True),
    HandoverChecklistItem("Internal owner knows how to restart the index refresh", "internal owner demo", done=True),
    HandoverChecklistItem("Internal owner can add a document to the knowledge base", "internal owner demo", done=False),
    HandoverChecklistItem("Monitoring alert fires to internal owner (not FDE)", "test alert sent", done=True),
    HandoverChecklistItem("Handover document reviewed and signed off by internal owner", "written sign-off", done=True),
    HandoverChecklistItem("Success metrics dashboard accessible to internal owner", "internal owner demo", done=True),
    HandoverChecklistItem("Escalation path tested (FDE reachable in first 90 days)", "test message sent", done=False),
]

run_handover_checklist(STANDARD_CHECKLIST)

## 3. CinemaStream in Practice — On Loan at FilmiBox

In [ ]:
# filmibox/handover/adoption_plan.py

from dataclasses import dataclass, field
from typing import List

@dataclass
class TrainingScenario:
    scenario_name: str
    role: str
    setup: str
    steps: List[str]
    common_failure: str
    success_signal: str

@dataclass
class TrainingPlan:
    tool_name: str
    audience: str
    duration_minutes: int
    scenarios: List[TrainingScenario]
    self_practice_task: str
    owner_contact: str

@dataclass
class AdoptionMetric:
    metric_name: str
    definition: str
    baseline: float
    target: float
    target_date: str
    current: float = 0.0

@dataclass
class AdoptionPlan:
    tool_name: str
    launch_date: str
    metrics: List[AdoptionMetric]
    review_cadence: str

@dataclass
class HandoverChecklistItem:
    item: str
    verified_by: str
    done: bool = False

@dataclass
class HandoverDocument:
    system_name: str
    engagement_end_date: str
    internal_owner: str
    escalation_contact: str
    what_it_does: str
    what_it_does_not_do: str
    how_to_check_health: str
    how_to_restart: str
    known_limitations: str
    checklist: List[HandoverChecklistItem] = field(default_factory=list)

def check_adoption_status(plan: AdoptionPlan) -> None:
    print(f"Adoption status: {plan.tool_name}")
    print(f"Review cadence: {plan.review_cadence}")
    print()
    for m in plan.metrics:
        if m.target == m.baseline:
            pct = 100.0
        else:
            pct = min(100.0, (m.current - m.baseline) / (m.target - m.baseline) * 100)
        status = "✓ ON TRACK" if pct >= 70 else "⚠ AT RISK" if pct >= 40 else "✗ STALLED"
        print(f"{status}  {m.metric_name}: {m.current} (target {m.target} by {m.target_date})")

def run_handover_checklist(doc: HandoverDocument) -> None:
    done = sum(1 for i in doc.checklist if i.done)
    total = len(doc.checklist)
    print(f"Handover: {doc.system_name}")
    print(f"Owner: {doc.internal_owner} | Escalation: {doc.escalation_contact}")
    print(f"Checklist: {done}/{total}")
    print()
    for item in doc.checklist:
        symbol = "✓" if item.done else "✗"
        print(f"  [{symbol}] {item.item}")

def main():
    # --- Adoption metrics for the billing lookup tool (week-one data from pilot) ---
    billing_adoption = AdoptionPlan(
        tool_name="Billing Lookup Tool — FilmiBox",
        launch_date="2026-07-01",
        metrics=[
            AdoptionMetric(
                "Weekly active users (% of support team)",
                "Distinct support agents who ran ≥1 billing query this week / 5 total agents",
                baseline=0.0, target=0.80, target_date="2026-07-28",
                current=0.60,   # 3 of 5 agents in week 1 — pilot agents active, 2 new agents pending training
            ),
            AdoptionMetric(
                "In-tool resolution rate",
                "Billing queries resolved in-tool without escalation / total billing queries",
                baseline=0.0, target=0.70, target_date="2026-07-28",
                current=0.786,  # pilot result carried forward
            ),
            AdoptionMetric(
                "Manual billing ticket rate",
                "Billing tickets opened per week / total billing contacts received",
                baseline=1.0, target=0.30, target_date="2026-07-28",
                current=0.50,   # still half going to tickets — routing problem, not tool problem
            ),
        ],
        review_cadence="Weekly for first 30 days; Dev reviews and sends Anjali a one-line update",
    )

    print("=== ADOPTION STATUS ===")
    check_adoption_status(billing_adoption)

    # --- Handover document for the full FilmiBox assistant system ---
    filmibox_handover = HandoverDocument(
        system_name="FilmiBox Internal Assistant (billing + wiki + complaint themes)",
        engagement_end_date="2026-07-07",
        internal_owner="Dev (dev@filmibox.io)",
        escalation_contact="FDE reader (reader@cinemastream.com) — available for 90 days post-handover",
        what_it_does=(
            "Answers billing and subscription questions using the subscriber database (MCP tool). "
            "Answers new-hire policy and process questions from the internal wiki (RAG, bilingual). "
            "Surfaces top complaint themes from Zendesk tickets indexed overnight (RAG, last 7 days)."
        ),
        what_it_does_not_do=(
            "Does NOT answer questions about content licensing, contracts, or partner agreements. "
            "Does NOT answer HR or payroll questions. "
            "Does NOT give answers about individual employee accounts or salary. "
            "Does NOT have access to the founder MRR dashboard (Phase 2, not yet built). "
            "If a query falls outside billing/wiki/complaint-themes scope, the system will say so — "
            "direct the user to the relevant team channel."
        ),
        how_to_check_health=(
            "1. Open the assistant URL and type 'health check'. It returns index freshness timestamps. "
            "2. Zendesk index must show last-updated within 25 hours (refresh runs 02:00 IST nightly). "
            "3. Wiki index must show last-updated within 49 hours (refresh runs 03:00 IST every other night). "
            "4. If either index is stale: check the #data-pipeline-alerts Slack channel first."
        ),
        how_to_restart=(
            "Zendesk refresh: `ssh filmibox-data 'cd /opt/assistant && python refresh_tickets.py'`. "
            "Wiki refresh: same host, `python refresh_wiki.py`. "
            "Both scripts log to /var/log/assistant/. Check the last 50 lines if they fail."
        ),
        known_limitations=(
            "Billing tool accuracy: 78.6% resolution rate in pilot (14 test queries). "
            "Subscriber names with special characters (Devanagari, Tamil script) may not match exactly — use email. "
            "Wiki in Hindi: works for standard queries; highly technical or legal language may miss relevant docs. "
            "Complaint themes: Zendesk only — does not pull from email or phone records. "
            "System is not designed for subscriber-facing use — internal team only."
        ),
        checklist=[
            HandoverChecklistItem("Dev can log in and run a billing query", "Dev live demo", done=True),
            HandoverChecklistItem("Dev knows how to check index freshness", "Dev live demo", done=True),
            HandoverChecklistItem("Dev can trigger a manual wiki refresh", "Dev live demo", done=True),
            HandoverChecklistItem("Dev can add a document to the wiki knowledge base", "Dev live demo", done=False),
            HandoverChecklistItem("Monitoring alert goes to Dev's Slack, not FDE", "test alert verified", done=True),
            HandoverChecklistItem("Handover document read and signed by Dev", "written sign-off", done=True),
            HandoverChecklistItem("Anjali briefed on Phase 2 PoV and timeline", "Anjali confirmation", done=True),
            HandoverChecklistItem("FDE escalation path tested (Dev messaged, FDE replied)", "test message", done=False),
        ],
    )

    print()
    print("=== HANDOVER CHECKLIST ===")
    run_handover_checklist(filmibox_handover)
    print()
    print("What this system does:")
    print(filmibox_handover.what_it_does)
    print()
    print("What it does NOT do:")
    print(filmibox_handover.what_it_does_not_do)
    print()
    print("Known limitations:")
    print(filmibox_handover.known_limitations)

if __name__ == "__main__":
    main()

In [ ]:
# billing_adoption + check_adoption_status are the deliverable (see adoption_plan.py)
from filmibox.handover.adoption_plan import build_billing_adoption, check_adoption_status
billing_adoption = build_billing_adoption()

# The at-risk metric resolved — update it after week 2 data:
billing_adoption.metrics[2].current = 0.25   # manual ticket rate dropped below target

# Re-check — now all on track:
check_adoption_status(billing_adoption)

## 4. Pitfalls & Pro Tips

## 5. Exercises